In [27]:
import torch
import numpy as np
import random
import torchaudio
import os
import glob
from pathlib import Path
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# --- SET YOUR KAGGLE PATHS ---
INPUT_BASE = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')


def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # If using GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forces deterministic algorithms
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False

# Execute immediately at the top of the script
seed_everything(42)

In [29]:
def generate_synthetic_dataset(stems_dir, noise_dir, output_dir, samples_per_genre=50, target_sr=22050, duration=30):
    """Generates deterministic noisy mashups and saves them to /kaggle/working/."""
    genres = ["blues", "classical", "country", "disco", "hiphop",
"jazz", "metal", "pop", "reggae", "rock"
]
    target_length = target_sr * duration
    
    # Get noise files from read-only input
    noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)
    
    for genre in genres:
        # Create output directories in the writable /kaggle/working/ directory
        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)
        
        song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
        if not song_folders: 
            print(f"Warning: No songs found for genre {genre}")
            continue
        
        for i in range(samples_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stems = []
            stem_types = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
            
            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if os.path.exists(stem_path):
                    waveform, sr = torchaudio.load(stem_path)
                    
                    # Basic Resampling check (if needed)
                    if sr != target_sr:
                        resampler = torchaudio.transforms.Resample(sr, target_sr)
                        waveform = resampler(waveform)

                    if waveform.shape[1] > target_length:
                        waveform = waveform[:, :target_length]
                    elif waveform.shape[1] < target_length:
                        waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
                    stems.append(waveform)
            
            if len(stems) == 4:
                mashup = torch.stack(stems).sum(dim=0)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                noise_file = random.choice(noise_files)
                noise, _ = torchaudio.load(noise_file)
                
                if noise.shape[1] > target_length:
                    noise = noise[:, :target_length]
                    
                start_idx = random.randint(0, target_length - noise.shape[1])
                intensity = random.uniform(0.1, 0.4)
                
                mashup[:, start_idx:start_idx + noise.shape[1]] += (noise * intensity)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                # Save to /kaggle/working/
                out_path = genre_out_dir / f"mashup_{i:03d}.wav"
                torchaudio.save(str(out_path), mashup, target_sr)

# Run the generation
generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=50)

In [30]:
import os
import glob
import torch
import torchaudio
from pathlib import Path

def extract_and_save_features(input_dir, output_dir, target_sr=22050):
    """Converts audio to Mel-spectrograms in dB and saves as PyTorch tensors."""
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=target_sr, n_fft=2048, hop_length=512, n_mels=128
    )
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    # Find all .wav files in the input directory
    wav_files = glob.glob(os.path.join(input_dir, '**', '*.wav'), recursive=True)
    
    if not wav_files:
        print(f"Warning: No .wav files found in {input_dir}")
        return

    for wav_path in wav_files:
        # Replicate directory structure
        rel_path = os.path.relpath(wav_path, input_dir)
        out_path = Path(output_dir) / rel_path
        out_path = out_path.with_suffix('.pt')
        
        # Ensure the target directory exists in /kaggle/working/
        out_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Process and save
        waveform, sr = torchaudio.load(wav_path)
        mel_spec = mel_transform(waveform)
        mel_spec_db = amplitude_to_db(mel_spec)
        
        torch.save(mel_spec_db, out_path)
    
    print(f"Successfully saved {len(wav_files)} feature files to {output_dir}")


INPUT_DIR = '/kaggle/working/synthetic_mashups/train'
OUTPUT_DIR = '/kaggle/working/features/train'

extract_and_save_features(INPUT_DIR, OUTPUT_DIR)

Successfully saved 500 feature files to /kaggle/working/features/train


In [2]:
import torch
import torch.nn as nn
import glob
import os
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

class PrecomputedFeatureDataset(Dataset):
    def __init__(self, features_dir):
        self.files = glob.glob(os.path.join(features_dir, '**', '*.pt'), recursive=True)
        self.genres = sorted(['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock'])
        self.genre_to_idx = {g: i for i, g in enumerate(self.genres)}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        # Extract genre from the directory name
        genre = Path(file_path).parent.name
        label = self.genre_to_idx[genre] # converts genre name to a numerically encoded value
        
        # Load precomputed tensor
        feature = torch.load(file_path)
        return feature, label



## Q1) You ran the generate_synthetic_dataset script with samples_per_genre=50. If the script executed successfully across all 10 target genres, exactly how many .wav files should now exist in your /kaggle/working/synthetic_mashups/train/ directory tree?

In [3]:
import os

base_dir = "/kaggle/working/synthetic_mashups/train/"

count = 0
for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.endswith(".wav"):
            count += 1

print("Total .wav files:", count)

Total .wav files: 500


## Q2) Load any of your newly generated .wav files using torchaudio.load(). Given our configuration (target_sr=22050, duration=30), what is the exact tensor shape of the resulting waveform?
Provide in tuple format, example (x,y)

In [17]:
base_dir = "/kaggle/working/synthetic_mashups/train/blues/mashup_000.wav"

waveform, sr = torchaudio.load(base_dir)

print("Sample rate:", sr)
print("Waveform shape:", tuple(waveform.shape))

Sample rate: 22050
Waveform shape: (2, 661500)


## Using the extract_and_save_features script with n_fft=2048, hop_length=512, and n_mels=128, you converted the waveforms into .pt files. 

If you load one of these pre-computed tensors using torch.load(), what is its exact dimension shape?

In [18]:
feature_dir = "/kaggle/working/features/train/blues/mashup_000.pt"

# load tensor
features = torch.load(feature_dir)

print("Tensor shape:", tuple(features.shape))

Tensor shape: (2, 128, 1292)


## Build Your Model (The CRNN)
Goal: You will build a model that understands both what sounds are playing (using a CNN) and the rhythm of how they play over time (using an RNN).

What to Code:

Create a PyTorch class called CRNN with these exact pieces:

1. The Input Your model will take in a 1-channel Mel-spectrogram (treat it like a single-color image).

2. The CNN (Sound Feature Extractor)

Write two blocks of layers to find patterns in the audio frequencies:

Block 1: Conv2D (32 filters, $3 \times 3$ kernel, padding 1) $\rightarrow$ BatchNorm2d $\rightarrow$ ReLU $\rightarrow$ MaxPool2d ($2 \times 2$ window).

Block 2: Conv2D (64 filters, $3 \times 3$ kernel, padding 1) ---> BatchNorm2d ---> ReLU ---> MaxPool2d ($2 \times 2$ window).

3. The Bridge (Reshaping)

The CNN outputs a 3D block of data (channels, frequencies, and time). To pass this into an RNN, you must reshape it. Keep the time steps as your sequence, and flatten the channels and frequencies together into one single feature list per time step.

4. The RNN (Rhythm Tracker)

Pass your reshaped sequence into a 1-layer Bidirectional LSTM. (Set hidden_size=64 and batch_first=True).

5. The Final Prediction

The LSTM outputs data for every single time step. To get one final answer for the whole 30-second song, take the maximum value across all time steps (known as Global Max Pooling). Finally, pass that summary vector through a standard Linear layer to output the 10 genre scores.

In [19]:
import torch
import torch.nn as nn

class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        
        # TODO 1: Define the CNN Backbone using nn.Sequential
        # Expect an input of shape (Batch_Size, 1, 128, Time_Steps)
        # Block 1: Conv2d(1 -> 32, kernel=3, padding=1) -> BatchNorm2d -> ReLU -> MaxPool2d(2)
        # Block 2: Conv2d(32 -> 64, kernel=3, padding=1) -> BatchNorm2d -> ReLU -> MaxPool2d(2)
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        # TODO 2: Define the RNN Component
        # Hint: Calculate the flattened feature size coming out of your CNN.
        # Original Mels = 128. After two MaxPool2d(2) layers, Mels = 128 / 4 = 32.
        # Channels = 64. So, Input Size = Channels * Mels = 2048.
        # Create a 1-layer Bidirectional LSTM with hidden_size=64 and batch_first=True.
        self.lstm = nn.LSTM(
            input_size=2048,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        
        # TODO 3: Define the Classifier
        # Create a Fully Connected (Linear) layer. 
        # Hint: Since the LSTM is bidirectional, the input features will be hidden_size * 2.
        self.fc = nn.Linear(64 * 2, num_classes)

    def forward(self, x):
        """
        Input:
            x: Tensor of shape (Batch, 1, 128, Time) representing Mel-spectrograms.
        Output:
            logits: Tensor of shape (Batch, num_classes) representing unnormalized class scores.
        """
        
        # TODO 4: Pass 'x' through the CNN backbone
        # Expected shape after CNN: (Batch, Channels=64, Mels=32, Time)
        x = self.cnn(x)
        
        # TODO 5: Bridge the gap (Shape Manipulation)
        # Permute and reshape the CNN output to be compatible with the LSTM.
        # Extract b, c, f, t from the tensor shape.
        b, c, f, t = x.shape
        
        # Permute so Time becomes second dimension
        x = x.permute(0, 3, 1, 2)
        
        # Flatten Channels and Mels
        x = x.reshape(b, t, c * f)
        
        # TODO 6: Pass the reshaped sequence through the LSTM
        # The LSTM returns output and (hidden_state, cell_state). You only need the output.
        x, _ = self.lstm(x)
        
        # TODO 7: Global Max Pooling over the time dimension
        # Collapse the sequence down to a single vector using torch.max() over time
        x, _ = torch.max(x, dim=1)
        
        # TODO 8: Pass the pooled vector through the fully connected layer
        logits = self.fc(x)
        
        return logits

## Q4) Inside the CRNN model's forward pass, the data moves from the CNN backbone to the LSTM. If your input batch has a size of 32, what is the exact shape of the tensor immediately after the second MaxPool2d(2) layer, right before the .permute() operation?

In [20]:
cnn = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, padding=1),
    nn.BatchNorm2d(32),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(2)
)

# Dummy input (Batch=32, Channels=1, Mels=128, Time=1292)
x = torch.randn(32, 1, 128, 1292)

# Pass through CNN
out = cnn(x)

print("Shape after CNN:", out.shape)

Shape after CNN: torch.Size([32, 64, 32, 323])


## Q5) Your model uses a Bidirectional LSTM with input_size=2048 and hidden_size=64. Write a small snippet using sum(p.numel() for p in model.lstm.parameters() if p.requires_grad) to calculate the exact number of trainable parameters in the LSTM layer alone. What is that integer value?

In [21]:
# Create the LSTM layer exactly like in the CRNN
lstm = nn.LSTM(
    input_size=2048,
    hidden_size=64,
    num_layers=1,
    batch_first=True,
    bidirectional=True
)

# Count trainable parameters
num_params = sum(p.numel() for p in lstm.parameters() if p.requires_grad)

print(num_params)

1082368


# training model

In [25]:


# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model
model = CRNN(num_classes=10).to(device)

# loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
train_dataset = PrecomputedFeatureDataset("/kaggle/working/features/train")
# dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

num_epochs = 10

In [ ]:
# from sklearn.metrics import f1_score
# for epoch in range(num_epochs):

#     model.train()
#     total_loss = 0

#     all_preds = []
#     all_labels = []

#     for x, y in train_loader:

#         x = x.to(device)
#         y = y.to(device)

#         optimizer.zero_grad()

#         outputs = model(x)

#         loss = criterion(outputs, y)
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()

#         preds = torch.argmax(outputs, dim=1)

#         all_preds.extend(preds.cpu().numpy())
#         all_labels.extend(y.cpu().numpy())

#     avg_loss = total_loss / len(train_loader)
#     f1 = f1_score(all_labels, all_preds, average="macro")

#     print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_loss:.4f} | F1 Score: {f1:.4f}")

## Q6) When applying a 2D CNN to a Mel-spectrogram, translation invariance along the X-axis (Time) is highly beneficial, as a guitar solo at 0:15 is the same as a guitar solo at 0:25. However, why is translation invariance along the Y-axis (Mel-Frequency Bins) conceptually problematic for music?


- Moving along the Y-axis changes the tempo of the song.
- Moving a pattern up the Y-axis fundamentally changes its pitch and instrument identity (e.g., shifting a low bassline up 80 bins turns it into a high-pitched synth).
- The Y-axis represents the stereo panning (left/right speakers).
- The CNN kernel cannot mathematically move vertically across a spectrogram.
